# n1 · 标量自动微分 Scalar Autograd

> **能力标签**：SH8（PyTorch 基础 / PyTorch fundamentals）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释**计算图**（computation graph）与**链式法则**（chain rule）的关系。
2. 实现一个支持加法（`+`）和乘法（`*`）的 `Value` 类，记录运算历史。
3. 用**拓扑排序**（topological sort）完成**反向传播**（backpropagation），自动计算所有叶节点的梯度（gradient）。
4. 用有限差分（finite difference）验证梯度的数值正确性。

> 对应能力：**SH8**


## 概念讲解 Concepts

### 1. 为什么需要自动微分 Why Autograd?

深度学习的训练核心是**梯度下降**（gradient descent）：

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_{\theta} L$$

其中 $L$ 是损失函数，$\nabla_{\theta} L$ 是损失对所有参数的梯度。
当网络有数百万参数时，手动推导梯度不现实——**自动微分**（automatic differentiation, autograd）就是解决方案。

---

### 2. 计算图 Computation Graph

每次运算都产生一个**节点**，节点之间的边记录了"谁是谁的输入"。

```
a = Value(2.0)   b = Value(3.0)
          \           /
           c = a * b        # c.data = 6.0
               |
           d = c + a        # d.data = 8.0
```

图中每个节点存储：
- `data`：当前值
- `grad`：损失对该节点的梯度（初始为 0）
- `_backward`：该节点知道如何把梯度传回给自己的输入

---

### 3. 链式法则 Chain Rule

对于复合函数 $d = f(c) = c + a$，其中 $c = g(a, b) = a \cdot b$：

$$\frac{\partial d}{\partial a} = \frac{\partial d}{\partial c} \cdot \frac{\partial c}{\partial a} + \frac{\partial d}{\partial a}\bigg|_{\text{direct}}$$

**链式法则**将复杂导数分解为沿路径的局部导数的乘积，从输出节点向输入节点逐层传递。

---

### 4. 拓扑排序反向传播 Topological Backprop

反向传播需要**按逆拓扑序**遍历计算图，保证每个节点在其所有下游节点之前接收完整梯度。

```
正向：a, b → c → d   （叶→根）
反向：d → c → a, b   （根→叶，逆序）
```

算法：
1. 从输出节点出发，DFS 收集拓扑序列。
2. 把输出节点的 `grad` 设为 1.0（$\frac{\partial L}{\partial L} = 1$）。
3. 按逆序调用每个节点的 `_backward()`，将梯度"分发"给子节点。

---

### 5. 加法与乘法的局部导数 Local Gradients for + and *

| 运算 | 输出 | $\frac{\partial \text{out}}{\partial \text{self}}$ | $\frac{\partial \text{out}}{\partial \text{other}}$ |
|------|------|------|------|
| `self + other` | `self.data + other.data` | 1 | 1 |
| `self * other` | `self.data * other.data` | `other.data` | `self.data` |

梯度**累加**（`+=`）是关键：同一个节点可能在多条路径上，必须把所有路径的贡献相加。


## 示例 Worked Example

In [ ]:
# ── 完整 Value 类：支持 + 与 * ──────────────────────────────────────────────

class Value:
    """标量值节点，支持自动微分 (Scalar value node with autograd)."""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0          # 梯度（初始为 0）
        self._backward = lambda: None   # 默认无操作
        self._prev = set(_children)     # 父节点集合
        self._op = _op                  # 产生此节点的运算符（调试用）

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ── 加法 Addition ────────────────────────────────────────────────────────
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # d(out)/d(self) = 1, d(out)/d(other) = 1
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    # ── 乘法 Multiplication ──────────────────────────────────────────────────
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # d(out)/d(self)  = other.data
            # d(out)/d(other) = self.data
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad

        out._backward = _backward
        return out

    # ── 反射运算符（支持 2 + a 等写法）─────────────────────────────────────
    __radd__ = __add__
    __rmul__ = __mul__

    # ── 反向传播 Backpropagation ─────────────────────────────────────────────
    def backward(self):
        """从本节点出发，反向传播梯度到所有叶节点。"""
        # Step 1: 拓扑排序（DFS 后序）
        topo, visited = [], set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)

        # Step 2: 输出节点梯度 = 1
        self.grad = 1.0

        # Step 3: 按逆拓扑序传播
        for v in reversed(topo):
            v._backward()


# ── 示例计算：d = a * b + a ───────────────────────────────────────────────
a = Value(2.0)
b = Value(3.0)

c = a * b      # c = 6.0
d = c + a      # d = 8.0

print("正向传播 Forward pass:")
print(f"  a = {a.data}, b = {b.data}")
print(f"  c = a * b = {c.data}")
print(f"  d = c + a = {d.data}")

d.backward()

print()
print("反向传播 Backward pass:")
print(f"  d.grad = {d.grad:.4f}  (应为 1.0)")
print(f"  c.grad = {c.grad:.4f}  (应为 1.0, d/dc = 1)")
print(f"  a.grad = {a.grad:.4f}  (应为 4.0, 两条路径: via c → b.data=3, via d → 1)")
print(f"  b.grad = {b.grad:.4f}  (应为 2.0, via c → a.data=2)")


In [ ]:
# ── 有限差分梯度验证 Finite-Difference Gradient Check ──────────────────────
# 数值导数：(f(x+h) - f(x-h)) / (2h)  ≈  df/dx

def fd_grad(fn, x, h=1e-5):
    """中心差分估计 df/dx."""
    return (fn(x + h) - fn(x - h)) / (2 * h)


def make_expr(a_val, b_val):
    """重建计算图并返回 d = a*b + a."""
    a = Value(a_val)
    b = Value(b_val)
    d = a * b + a
    return a, b, d


# 验证 dd/da
def expr_a(a_val):
    _, _, d = make_expr(a_val, 3.0)
    return d.data

# 验证 dd/db
def expr_b(b_val):
    _, _, d = make_expr(2.0, b_val)
    return d.data


a, b, d = make_expr(2.0, 3.0)
d.backward()

fd_a = fd_grad(expr_a, 2.0)
fd_b = fd_grad(expr_b, 3.0)

print("梯度对比 Gradient Comparison (d = a*b + a, a=2, b=3):")
print(f"  dd/da: autograd = {a.grad:.6f} | finite-diff = {fd_a:.6f} | 误差 = {abs(a.grad - fd_a):.2e}")
print(f"  dd/db: autograd = {b.grad:.6f} | finite-diff = {fd_b:.6f} | 误差 = {abs(b.grad - fd_b):.2e}")
print()
assert abs(a.grad - fd_a) < 1e-8, "dd/da 不匹配！"
assert abs(b.grad - fd_b) < 1e-8, "dd/db 不匹配！"
print("✓ 梯度验证通过 Gradient check passed")


## 动手 Exercise

In [ ]:
# ── 动手练习：验证更复杂的表达式 ────────────────────────────────────────────
# 计算 f = (a + b) * (a + 1), 其中 a=2, b=1
# 手工推导：
#   f = (a+b)(a+1) = a^2 + a + ab + b
#   df/da = 2a + 1 + b = 2*2 + 1 + 1 = 6
#   df/db = a = 2

a = Value(2.0)
b = Value(1.0)
f = (a + b) * (a + Value(1.0))
f.backward()

print("f = (a+b)*(a+1),  a=2, b=1")
print(f"  f.data = {f.data:.4f}  (应为 9.0)")
print(f"  a.grad = {a.grad:.4f}  (应为 6.0)")
print(f"  b.grad = {b.grad:.4f}  (应为 2.0)")

# 有限差分验证
def f_a(av): a2 = Value(av); b2 = Value(1.0); r = (a2+b2)*(a2+Value(1.0)); return r.data
def f_b(bv): a2 = Value(2.0); b2 = Value(bv); r = (a2+b2)*(a2+Value(1.0)); return r.data

fd_a = (f_a(2.0+1e-5) - f_a(2.0-1e-5)) / 2e-5
fd_b = (f_b(1.0+1e-5) - f_b(1.0-1e-5)) / 2e-5
print()
print(f"  df/da finite-diff = {fd_a:.6f}  (应与 a.grad 一致)")
print(f"  df/db finite-diff = {fd_b:.6f}  (应与 b.grad 一致)")
assert abs(a.grad - fd_a) < 1e-6
assert abs(b.grad - fd_b) < 1e-6
print("✓ 练习验证通过")


## 延伸阅读 Further Reading

1. **Andrej Karpathy «micrograd»**：<https://github.com/karpathy/micrograd> — 本课的原型；仅 150 行 Python 实现完整 autograd 引擎。
2. **CS231n Backpropagation Notes**：<https://cs231n.github.io/optimization-2/> — 斯坦福经典反向传播笔记，含直觉图解。
3. **Olah (2015) "Calculus on Computational Graphs"**：<https://colah.github.io/posts/2015-08-Backprop/> — 强烈推荐的入门博文。
4. **PyTorch Autograd Mechanics**：<https://pytorch.org/docs/stable/notes/autograd.html> — 工业级 autograd 的实现文档。


## 关联练习 Related Assignments

```bash
python check.py nanograd-l1
```

> 实现 `Value.__add__`、`Value.__mul__` 与 `Value.backward`，使自动微分在加法与乘法上正确计算梯度。

> 能力标签：**SH8** · threshold ≥ 0.7
